<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>
<p><font size="5" color='grey'> <b>
Large Language Model - Abfrage auf Datenbanken

---

# **0 <font color='orange'>|</font> Install & Import**
---

In [ ]:
!pip install -qU langchain sqlalchemy
!pip install -qU langchain-experimental
!pip install -qU langchain_openai
!pip install -qU gradio

In [ ]:
# Hilfslösung für Fehler mit OpenAI 1.56.0 - Client.__init__() hat ein unerwartetes Schlüsselwortargument 'proxyes' erhalten
!pip install openai==1.55.3 httpx==0.27.2 --force-reinstall --quiet
import os

os.kill(os.getpid(), 9)

In [ ]:
from langchain_experimental.sql.base import SQLDatabase
from langchain_experimental.sql.base import SQLDatabaseChain
from langchain_openai import ChatOpenAI
import os
from typing import Any
from pydantic import BaseModel
from google.colab import userdata
import sqlite3
import gradio as gr

# **1 <font color='orange'>|</font> Konstanten**
---



In [ ]:
# Set your OpenAI API key
OPENAI_API_KEY = userdata.get("OpenAI-API-Key")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # Replace with your actual key

In [ ]:
# Initialize the LLM
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# **2 <font color='orange'>|</font> Datenbank laden**

---

In [ ]:
# SQLite-Datenbank laden
db_path = "/content/chinook.db"
db = SQLDatabase.from_uri("sqlite:////content/chinook.db")

# **3 <font color='orange'>|</font> SQLDatabaseChain**
---



In [ ]:
# Definiere eine leere Callbacks-Klasse; Vermeidung Fehlermeldung
class Callbacks(BaseModel):
    pass


BaseCache = Callbacks
SQLDatabaseChain.model_rebuild()

# Datenbank-Kette
db_chain = SQLDatabaseChain(llm=llm, database=db, verbose=False)

# **4 <font color='orange'>|</font> Datenbank-Kontext**
---


In [ ]:
def get_table_info(db_path):
    """Retrieves table and column information from the database."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [table[0] for table in cursor.fetchall()]

    # Get column names for each table
    table_info = {}
    for table_name in tables:
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = [column[1] for column in cursor.fetchall()]
        table_info[table_name] = columns

    conn.close()
    return table_info

In [ ]:
table_info = get_table_info(db_path)

# **5 <font color='orange'>|</font> App-Integration Chatbot**
---

In [ ]:
def run_query_with_schema_check(
    query, db_chain=db_chain, db_path=db_path, table_info=table_info
):

    # Create a detailed schema description
    schema_info_str = "**Database Schema:**\n"
    for table_name, columns in table_info.items():
        schema_info_str += f"- **Table: {table_name}**\n"
        for column in columns:
            schema_info_str += f"  - {column}\n"

    # Modify the prompt, providing the schema information
    modified_prompt = f"{schema_info_str}\n\n**User Query:** {query}"

    try:
        response = db_chain.invoke(modified_prompt)
    except sqlite3.OperationalError as e:
        if "no such table" in str(e):
            response = f"Error: The query references a table that does not exist in the database. {schema_info_str}"
        elif "no such column" in str(e):
            response = f"Error: The query references a column that does not exist in the specified table. {schema_info_str}"
        else:
            response = f"Error executing query: {e}"
    return f"{response['result']}"

In [ ]:
def call_chatbot(query, chat_history=[]):
    result = run_query_with_schema_check(query)
    return result

# **6 <font color='orange'>|</font> App-Design**
---

In [ ]:
# Festlegen Design der App
demo = gr.ChatInterface(
    fn=call_chatbot,
    type="messages",
    examples=[
        "Who was the artist with the highest revenues?",
        "Who are the artis in in Genre Rock?",
        "What was the invoce with the highest total and who was the customer?",
    ],
    title="📚 Datenbank Chatbot",
    description="\n\n*Der Chatbot wertet die Datenbank Dokumente aus und beantwortet Fragen zum Inhalt*",
)

In [ ]:
demo.launch(share=True)